# Text Generation using Transformers

In [ ]:
import numpy as np 
import pandas as pd 
import os
import transformers 
from transformers import AutoTokenizer
import torch 
from torch.utils.data import DataLoader 
from datasets import load_dataset,DatasetDict
from transformers import AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from accelerate import Accelerator
from transformers import get_scheduler
from tqdm.notebook import tqdm

In [ ]:
train_ds = load_dataset("huggingface-course/codeparrot-ds-train",split='train')
test_ds = load_dataset("huggingface-course/codeparrot-ds-valid",split='validation')
raw_dataset = DatasetDict(
    {
        "train" : train_ds.shuffle().select(range(50000)),
        "validate" : test_ds.shuffle().select(range(1000))
    }
)


In [ ]:
print("Data Overview")
print("-"*100)
print(raw_dataset['train'][1]['content'][:200])
print("-"*100)

In [ ]:
checkpoint = ''
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")

**Preprocessing the dataset**

In [ ]:
context_length = 128
def tokenize_function(example):
    outputs = tokenizer(
        example['content'],
        truncation = True,
        max_length = context_length,
        return_overflowing_tokens = True,
        return_length = True
    )
    input_batch = []
    for length,input_ids in zip(outputs['length'],outputs['input_ids']):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids":input_batch}

In [ ]:
tokenized_dataset = raw_dataset.map(tokenize_function,batched=True,remove_columns=raw_dataset['train'].column_names)
tokenized_dataset

In [ ]:
from transformers import GPT2LMHeadModel,AutoConfig

config = AutoConfig.from_pretrained(
    pretrained_model_name_or_path = "gpt2",
    vocab_size = len(tokenizer),
    n_ctx = context_length,
    bos_token_id = tokenizer.bos_token_id,
    eos_token_id = tokenizer.bos_token_id
)
model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 parameters : {model_size/1000**2:.1f}M Paramters")

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

In [ ]:
def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
    # Shift so that tokens < n predict n
    shift_labels = inputs[..., 1:].contiguous()
    shift_logits = logits[..., :-1, :].contiguous()
    # Calculate per-token loss
    loss_fct = torch.nn.CrossEntropyLoss(reduce=False)
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    # Resize and average loss per sample
    loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
    # Calculate and scale weighting
    weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(
        axis=[0, 2]
    )
    weights = alpha * (1.0 + weights)
    # Calculate weighted average
    weighted_loss = (loss_per_sample * weights).mean()
    return weighted_loss

In [ ]:
weight_decay = 0.1
def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [
        {"params": params_with_wd, "weight_decay": weight_decay},
        {"params": params_without_wd, "weight_decay": 0.0},
    ]

In [ ]:
def evaluate():
    model.eval()
    losses = []
    for step, batch in enumerate(Eval_dataloader):
        with torch.no_grad():
            outputs = model(batch["input_ids"], labels=batch["input_ids"])

        losses.append(accelerator.gather(outputs.loss))
    loss = torch.mean(torch.cat(losses))
    try:
        perplexity = torch.exp(loss)
    except OverflowError:
        perplexity = float("inf")
    return loss.item(), perplexity.item()

In [ ]:
Batch_size = 16
tokenized_dataset.set_format('torch')
Train_dataloader = DataLoader(
    tokenized_dataset['train'],
    shuffle=True,
    batch_size=Batch_size,
    collate_fn = data_collator,
)

Eval_dataloader = DataLoader(
    tokenized_dataset['validate'],
    shuffle=False,
    batch_size = Batch_size,
    collate_fn = data_collator,
)

In [ ]:
Epochs = 1
optimizer = torch.optim.AdamW(get_grouped_params(model),lr=0.0002)
num_update_step_per_epoch = len(Train_dataloader)
num_train_steps = Epochs * num_update_step_per_epoch
model = model

lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_training_steps = num_train_steps,
    num_warmup_steps = 1
)

accelerator = Accelerator()
model,optimizer,Train_dataloader,Eval_dataloader = accelerator.prepare(model,optimizer,Train_dataloader,Eval_dataloader)

In [ ]:
keytoken_ids = []
for keyword in [
    "plt",
    "pd",
    "sk",
    "fit",
    "predict",
    " plt",
    " pd",
    " sk",
    " fit",
    " predict",
    "testtest",
]:
    ids = tokenizer([keyword]).input_ids[0]
    if len(ids) == 1:
        keytoken_ids.append(ids[0])
    else:
        print(f"Keyword has not single token: {keyword}")
print(keytoken_ids)

In [ ]:
gradient_acc_step = 8
eval_steps = 5_000

model.train()
completed_steps = 0
for epoch in range(Epochs):
    for step,batch in tqdm(enumerate(Train_dataloader,start=1),total=num_train_steps):
        logits = model(batch['input_ids']).logits
        loss = keytoken_weighted_loss(batch["input_ids"], logits, keytoken_ids)
        if step % 100 ==0:
            accelerator.print(
                {
                    "steps": completed_steps,
                    "loss/train": loss.item() * gradient_acc_step,
                }
            )
        loss = loss / gradient_acc_step
        accelerator.backward(loss)
        if (step % gradient_acc_step == 0):
            accelerator.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            completed_steps = completed_steps + 1 

        if (step % (eval_steps*gradient_acc_step)) == 0:
            eval_loss , perplexity = evaluate()
            accelerator.print({"loss/eval": eval_loss, "perplexity": perplexity})
            model.train()
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained("/kaggle/working/Models/", save_function=accelerator.save)